In [24]:
import rosbag
import numpy as np
from nav_msgs.msg import Odometry
import tf
import matplotlib.pyplot as plt

def extract_odometry_data(bag_file, topic_names):
    """
    Extracts x, y, and yaw data from specified odometry topics in a bag file.

    Args:
        bag_file (str): Path to the ROS bag file.
        topic_names (list): List of odometry topic names to extract.

    Returns:
        dict: A dictionary with topic names as keys and numpy arrays of shape (n, 3) as values.
    """
    # Dictionary to store data for each topic
    odometry_data = {topic: [] for topic in topic_names}

    # Open the bag file
    with rosbag.Bag(bag_file, 'r') as bag:
        for topic, msg, t in bag.read_messages(topics=topic_names):
            if topic in topic_names:
                # Extract position (x, y)
                x = msg.pose.pose.position.x
                y = msg.pose.pose.position.y

                # Extract orientation (yaw)
                orientation_q = msg.pose.pose.orientation
                orientation_list = [orientation_q.x, orientation_q.y, orientation_q.z, orientation_q.w]
                _, _, yaw = tf.transformations.euler_from_quaternion(orientation_list)

                # Append the data
                odometry_data[topic].append([x, y, yaw])

    # Convert lists to numpy arrays
    for topic in topic_names:
        odometry_data[topic] = np.array(odometry_data[topic])

    return odometry_data

def compute_transformation(source, target):
    """
    Computes the rigid transformation (rotation + translation) from source to target.

    Args:
        source (array): Source coordinate as [x, y, yaw].
        target (array): Target coordinate as [x, y, yaw].

    Returns:
        tuple: Translation (dx, dy) and rotation (dtheta).
    """
    dx = target[0] - source[0]
    dy = target[1] - source[1]
    dtheta = target[2] - source[2]
    return dx, dy, dtheta

def apply_transformation(data, dx, dy, dtheta):
    """
    Applies a rigid transformation to the given odometry data.

    Args:
        data (numpy array): Array of shape (n, 3) with [x, y, yaw].
        dx (float): Translation in x.
        dy (float): Translation in y.
        dtheta (float): Rotation angle in radians.

    Returns:
        numpy array: Transformed data of shape (n, 3).
    """
    transformed_data = []
    for x, y, yaw in data:
        # Rotate
        x_new = x * np.cos(dtheta) - y * np.sin(dtheta)
        y_new = x * np.sin(dtheta) + y * np.cos(dtheta)
        # Translate
        x_new += dx
        y_new += dy
        yaw_new = yaw + dtheta
        transformed_data.append([x_new, y_new, yaw_new])
    return np.array(transformed_data)

def plot_odometry_data(odometry_data):
    """
    Plots odometry data on an X-Y plane with yaw represented as arrows.

    Args:
        odometry_data (dict): A dictionary with topic names as keys and numpy arrays of shape (n, 3) as values.
    """
    plt.figure(figsize=(10, 8))

    for topic, data in odometry_data.items():
        x = data[:, 0]
        y = data[:, 1]
        yaw = data[:, 2]

        # Plot positions
        plt.scatter(x, y, label="{}".format(topic), s=10)

        # Plot yaw as arrows
        for i in range(0, len(x), max(1, len(x)//100)):  # Limit number of arrows for clarity
            dx = 0.1 * np.cos(yaw[i])
            dy = 0.1 * np.sin(yaw[i])
            plt.arrow(x[i], y[i], dx, dy, head_width=0.05, head_length=0.1, fc='k', ec='k')

    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title("Odometry Data Visualization")
    plt.legend()
    plt.grid()
    plt.axis('equal')
    plt.show()

if __name__ == "__main__":
    # Path to your bag file
    bag_file_path = "/home/aristos/catkin_ws/bags/fast_lio_odometry_comparison_full_2025-01-08-12-02-19.bag"

    # Topics to extract
    # odometry_topics = ["/aristos/ground_truth", "/Odometry"]
    odometry_topics = ["/aristos/odometry/navsat", "/aristos/ground_truth", "/aristos/odometry/onlyRTK"]


    # Extract data
    odometry_arrays = extract_odometry_data(bag_file_path, odometry_topics)

    # # Transform topic B to the frame of topic A
    # if len(odometry_topics) == 2:
    #     source_topic = odometry_topics[0]
    #     target_topic = odometry_topics[1]

    #     source_start = odometry_arrays[source_topic][0]
    #     target_start = odometry_arrays[target_topic][0]
    #     print(source_start)
    #     print(target_start)

    #     dx, dy, dtheta = compute_transformation(source_start, target_start)
    #     print(dx, dy, dtheta)
    #     odometry_arrays[target_topic] = apply_transformation(odometry_arrays[target_topic], -dx, -dy, -dtheta)

    # Plot data
    plt.figure(figsize=(10, 8))

    for topic, data in odometry_arrays.items():
        x = data[:, 0]
        y = data[:, 1]
        yaw = data[:, 2]

        # Plot positions
        plt.scatter(x, y, label="{}".format(topic), s=10)

        # Plot yaw as arrows
        for i in range(0, len(x), max(1, len(x)//100)):  # Limit number of arrows for clarity
            dx = 0.1 * np.cos(yaw[i])
            dy = 0.1 * np.sin(yaw[i])
            plt.arrow(x[i], y[i], dx, dy, head_width=0.05, head_length=0.1, fc='k', ec='k')

    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title("Odometry Data Visualization")
    plt.legend()
    plt.grid()
    plt.axis('equal')
    plt.show()

IOError: [Errno 2] No such file or directory: '/home/aristos/catkin_ws/bags/fast_lio_odometry_comparison_full_2025-01-08-11-31-22.bag'

In [21]:
print(odometry_arrays[target_topic][0])

[ -1.36075254 -24.84151047   3.06930637]


In [13]:
compute_transformation(np.array([0.0,0.0,0.0]), np.array([1.0,2.0,0.5]))

(1.0, 2.0, 0.5)

In [15]:
apply_transformation(np.array([[0.0,0.0,0.0]]), 1.0, 2.0, 0.5)

array([[1. , 2. , 0.5]])

In [25]:
import numpy as np
import math

def compute_fitted_heading(fit_points, prev_cmd, initial_covariance = 0.1):
    """Compute heading using line fitting and correct it based on robot motion."""
    if len(fit_points) < 2:
        # self.fit_points = []
        return None, initial_covariance

    # Extract x and y coordinates
    x_vals = np.array([p[0] for p in fit_points])
    y_vals = np.array([p[1] for p in fit_points])

    # ----- Least Squares Fitting -----
    A = np.vstack([x_vals, np.ones(len(x_vals))]).T
    m, b = np.linalg.lstsq(A, y_vals, rcond=None)[0]  # Solve y = mx + b
    fitted_heading = math.atan(m)  # Convert slope to angle

    # ----- Compute Residuals for Covariance -----
    y_predicted = m * x_vals + b
    residuals = y_vals - y_predicted
    covariance = np.var(residuals) if len(residuals) > 1 else initial_covariance

    # ----- Correct Heading Based on Robot Motion -----
    # Determine the direction of motion using the first and last points
    x_first, y_first = fit_points[0]
    x_last, y_last = fit_points[-1]
    dx = x_last - x_first
    dy = y_last - y_first

    # Compute the displacement angle
    displacement_angle = math.atan2(dy, dx)

    # Check if the fitted heading aligns with the displacement direction
    angle_diff = abs(fitted_heading - displacement_angle)
    if angle_diff > math.pi / 2:  # Fitted heading is opposite to displacement direction
        fitted_heading += math.pi  # Flip by 180 degrees

    # Apply velocity sign correction
    if prev_cmd < 0:  # Robot moving backward
        fitted_heading += math.pi  # Flip by 180 degrees

    # Wrap heading to [-pi, pi]
    fitted_heading = (fitted_heading + math.pi) % (2 * math.pi) - math.pi
    # self.fit_points = []

    return fitted_heading, covariance


fit_points = [[3,3],[0,0],[1,1],[2,2]]
prev_cmd = -1

fitted_heading, covariance = compute_fitted_heading(fit_points, prev_cmd)
print("Heading: {}".format(fitted_heading*180.0/math.pi))
print("Covariance: {}".format(covariance))

Heading: 45.0
Covariance: 3.61854478158e-31


In [39]:
fit_points = np.array([[0,0],[1,1],[2,2],[0,0]])
fit_points_yaw = []
for pair in np.diff(fit_points, axis=0):
    fit_points_yaw.append(math.atan2(pair[0],pair[1]))

print(fit_points_yaw)
# max(np.diff(fit_points_yaw))

max(abs(np.diff(fit_points_yaw)))

[0.7853981633974483, 0.7853981633974483, -2.356194490192345]


3.141592653589793

In [48]:


from tf import transformations

q1 = transformations.quaternion_from_euler(0,0,0.17)
q2 = transformations.quaternion_from_euler(0,0,1.57)
q = transformations.quaternion_multiply(q1, q2)
transformations.euler_from_quaternion(q)

(0.0, -0.0, 1.74)

In [54]:
def angle_between_two_vectors(v1_start, v1_end, v2_start, v2_end):
    v1 = np.array([v1_end.x - v1_start.x, v1_end.y - v1_start.y])
    v2 = np.array([v2_end.x - v2_start.x, v2_end.y - v2_start.y])
    return np.math.atan2(np.linalg.det([v1,v2]),np.dot(v1,v2))

In [71]:
from geometry_msgs.msg import Point
v1_start = Point(x=1, y=1)
v1_end = Point(x=2, y=2)
v2_start = Point(x=3,y=3)
v2_end = Point(x=3,y=4)

angle_between_two_vectors(v1_start, v1_end, v2_start, v2_end)

0.7853981633974483

In [72]:
from collections import deque
d = deque()
d.append([v1_start, v2_start])
d.append([v1_end, v2_end])

In [70]:
from std_msgs.msg import Header

h = Header()
h.stamp.to_sec()

0.0

In [84]:
from geometry_msgs.msg import Point
from collections import deque

p1_1 = Point(x=1, y=1)
p1_2 = Point(x=2, y=2)
p1_3 = Point(x=3, y=3)

p2_1 = Point(x=1, y=0)
p2_2 = Point(x=2, y=0)
p2_3 = Point(x=3, y=0)

d = deque()
d.append([p1_1, p2_1])
d.append([p1_2, p2_2])
d.append([p1_3, p2_3])

In [77]:
import math
from geometry_msgs.msg import Point
from collections import deque

def euclidean_distance(p1, p2):
    return math.sqrt((p2.x - p1.x) ** 2 + (p2.y - p1.y) ** 2)

def total_distance(d):
    total_active = 0
    total_bootstrap = 0
    for i in range(len(d)-1):
        total_active += euclidean_distance(d[i][0], d[i + 1][0])
        total_bootstrap += euclidean_distance(d[i][1], d[i + 1][1])
    return total_active, total_bootstrap

total_distance(d)

[1.0, 2.0, 2.0]

In [87]:
d = deque()

d.append([np.array([p1_1.x, p1_1.y]), 
          np.array([p2_1.x, p2_1.y])])
d.append([np.array([p1_2.x, p1_2.y]), 
          np.array([p2_2.x, p2_2.y])])
d.append([np.array([p1_3.x, p1_3.y]), 
          np.array([p2_3.x, p2_3.y])])

In [103]:
def total_distances(d):

    arr = np.array(d)
    diffs = np.diff(arr, axis=0)
    distances = np.linalg.norm(diffs, axis=2) 
    return np.sum(distances, axis=0)


In [104]:
total_distances

array([2.82842712, 2.        ])